In [1]:
!pip install -q "ultralytics==8.4.149"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 6.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from zipfile import ZipFile
import torch

from ultralytics import SAM
import ultralytics

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

Ultralytics: 8.4.149
PyTorch: 2.11.0+cu128
Device: 0


**Paths**

In [6]:
DATASET_URL = (
    "https://github.com/ultralytics/"
    "assets/releases/download/v0.0.0/"
    "crack-seg.zip"
)

DATASETS_ROOT = Path("/content/datasets")
ARCHIVE_PATH = DATASETS_ROOT / "crack-seg.zip"
DATASET_ROOT = DATASETS_ROOT / "crack-seg"

In [ ]:
DATASETS_ROOT.mkdir(parents=True, exist_ok=True)

SPLITS = ("train", "val", "test")
EXPECTED_COUNTS = {
    "train": 3717,
    "val": 200,
    "test": 112,
}

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

print("Dataset root:", DATASET_ROOT)

Dataset root: /content/datasets/crack-seg


In [ ]:
dataset_ready = (DATASET_ROOT / "images" / "train").is_dir()

if not dataset_ready:
    if not ARCHIVE_PATH.is_file():
        print("Downloading Crack-Seg...")
        torch.hub.download_url_to_file(
            DATASET_URL, str(ARCHIVE_PATH), progress=True
        )
        
    print("Extracting dataset...")
    with ZipFile(ARCHIVE_PATH, "r") as zip_file:
        zip_file.extractall(DATASETS_ROOT)


if not DATASET_ROOT.is_dir():
    candidates = [
        path for path in DATASETS_ROOT.rglob("*")
        if path.is_dir() and (path / "images").is_dir() and "crack" in path.name.lower()
    ]

    if len(candidates) != 1:
        candidates = [
            path for path in DATASETS_ROOT.rglob("images")
            if path.is_dir()
        ]
        if len(candidates) == 1:
            candidates = [candidates[0].parent]

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not locate Crack-Seg root in {DATASETS_ROOT}. "
            f"Found candidates: {candidates}"
        )

    DATASET_ROOT = candidates[0]

100%|██████████| 91.6M/91.6M [00:01<00:00, 81.0MB/s]


Extracting dataset...


In [9]:
for split in SPLITS:
    required_directories = [
        (DATASET_ROOT / "images" / split),
        (DATASET_ROOT / "labels" / split)
    ]
    
    for directory in (required_directories):
        if not directory.is_dir():
            raise FileExistsError(directory)

print("Dataset extracted:", DATASET_ROOT)

Dataset extracted: /content/datasets


In [ ]:
VAL_IMAGES_DIR = DATASET_ROOT/ "images" / "val"
VAL_LABELS_DIR = DATASET_ROOT / "labels" / "val"

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "vision_unit_02_outputs/"
    "block_05/"
    "sam2_image"
)

FIGURE_DIR =  OUTPUT_ROOT / "figures"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


print("Validation images:", VAL_IMAGES_DIR)
print("Validation labels:", VAL_LABELS_DIR)
print("Outputs:", OUTPUT_ROOT)

Validation images: /content/datasets/images/val
Validation labels: /content/datasets/labels/val
Outputs: /content/drive/MyDrive/vision_unit_02_outputs/block_05/sam2_image


**Image index**

In [13]:
validation_image_index = {
    path.stem: path 
    for path in VAL_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
}

print("Validation images:", len(validation_image_index))

Validation images: 200


**Polygon → separate instance masks**

In [14]:
def load_instance_masks(image_path, label_path):
    bgr_image = cv2.imread(str(image_path))
    
    if bgr_image is None:
        raise ValueError(f"Cannot read: {image_path}")

    rgb_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    height, width = rgb_image.shape[:2]
    
    instance_masks = []
    
    label_text = label_path.read_text(encoding="utf-8").strip()
    if not label_text:
        return rgb_image, instance_masks
    
    for line_number, line in enumerate(label_text.splitlines(), start=1):
        values = line.split()
        
        if (len(values) < 7 or (len(values) - 1) % 2 != 0):
            raise ValueError(
                f"Invalid polygon: "
                f"{label_path.name}, "
                f"line {line_number}"
            )
        
        class_id = int(float(values[0]))
        if class_id != 0:
             raise ValueError(
                f"Unexpected class {class_id}"
            )
        
        normalized_points = np.asarray(values[1:], dtype=np.float32).reshape(-1, 2)
        
        if not np.all((normalized_points >= 0)& (normalized_points <= 1)):
            raise ValueError(
                f"Out-of-bounds polygon: "
                f"{label_path.name}"
            )
        
        pixel_points = np.empty_like(
            normalized_points, dtype=np.int32
        )
        
        pixel_points[:, 0] = np.clip(
            np.rint(normalized_points[:, 0] * width), 0, 
            width - 1
        ).astype(np.int32)
        
        pixel_points[:, 1] = np.clip(
            np.rint(normalized_points[:, 1] * height), 0, 
            height - 1
        ).astype(np.int32)
        
        mask = np.zeros((height, width), dtype=np.uint8)
        
        cv2.fillPoly(mask, [pixel_points], color=1)
        if mask.any():
            instance_masks.append(mask)
    
    return rgb_image, instance_masks

**Prompt and metric utilities**